# FieldLens report figures

Pilot disclaimer: FieldLens is a research pilot. Results are trends from small runs on a data subset, not benchmark numbers.

Set `PROFILE` below to the active scale profile (for example `pilot_v2`). This notebook only reads saved outputs. It does not train.

In [ ]:
from __future__ import annotations

import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

REPO = Path("..").resolve()
if not (REPO / "training").is_dir():
    REPO = Path(".").resolve()

PROFILE = os.environ.get("FIELDLENS_PROFILE", "pilot_v2")
RUNS = ["run1", "run2", "run3"]
DATA = REPO / "data" / "fieldlens" / PROFILE
STATS = DATA / "stats.json"
SPLIT = DATA / "split.csv"
SUBSET = DATA / "tiles"
OUT = REPO / "notebooks" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

ANOMALY = [
    "double_plant", "drydown", "endrow", "nutrient_deficiency",
    "planter_skip", "water", "waterway", "weed_cluster",
]
print("PROFILE", PROFILE)
print("STATS exists", STATS.is_file(), "path", STATS)

## Sample tiles (RGB, NIR, GT, valid mask)

In [ ]:
import sys
sys.path.insert(0, str(REPO / "training"))
from fieldlens.constants import CLASS_COLORS_RGB

rows = list(csv.DictReader(SPLIT.open())) if SPLIT.is_file() else []
sample = [r for r in rows if r["our_split"] == "test"][:4]

def load_valid(base, tid):
    b = np.array(Image.open(base / "boundaries" / f"{tid}.png").convert("L")) > 0
    m = np.array(Image.open(base / "masks" / f"{tid}.png").convert("L")) > 0
    return b & m

def gt_overlay(base, tid, valid):
    h, w = valid.shape
    out = np.zeros((h, w, 4), dtype=np.uint8)
    for c in ANOMALY:
        p = base / "labels" / c / f"{tid}.png"
        if not p.is_file():
            continue
        lab = np.array(Image.open(p).convert("L")) > 0
        out[lab & valid] = (*CLASS_COLORS_RGB[c], 200)
    return out

if sample:
    fig, axes = plt.subplots(len(sample), 4, figsize=(12, 3 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])
    for i, r in enumerate(sample):
        base = SUBSET / r["source_split"]
        tid = r["tile_id"]
        rgb = np.array(Image.open(base / "images" / "rgb" / f"{tid}.jpg").convert("RGB"))
        nir_p = base / "images" / "nir" / f"{tid}.jpg"
        if not nir_p.is_file():
            nir_p = base / "images" / "nir" / f"{tid}.png"
        nir = np.array(Image.open(nir_p).convert("L"))
        valid = load_valid(base, tid)
        gt = gt_overlay(base, tid, valid)
        axes[i, 0].imshow(rgb); axes[i, 0].set_title("RGB"); axes[i, 0].axis("off")
        axes[i, 1].imshow(nir, cmap="gray"); axes[i, 1].set_title("NIR"); axes[i, 1].axis("off")
        axes[i, 2].imshow(rgb); axes[i, 2].imshow(gt); axes[i, 2].set_title("GT"); axes[i, 2].axis("off")
        axes[i, 3].imshow(valid, cmap="gray"); axes[i, 3].set_title("valid"); axes[i, 3].axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "sample_tiles.png", dpi=140)
    plt.close(fig)
    print("wrote", OUT / "sample_tiles.png")
else:
    print("No tiles yet. Run extract for", PROFILE)

## Drone condition previews (severity 0 to 4)

In [ ]:
import torch
import yaml
from fieldlens.transforms_drone import apply_drone_condition

eval_cfg = yaml.safe_load((REPO / "training" / "configs" / "eval.yaml").read_text())

if sample:
    r = sample[0]
    base = SUBSET / r["source_split"]
    tid = r["tile_id"]
    rgb = np.array(Image.open(base / "images" / "rgb" / f"{tid}.jpg").convert("RGB"), dtype=np.float32) / 255.0
    x = torch.from_numpy(rgb).permute(2, 0, 1)
    effects = [
        ("motion_blur", "blur"),
        ("downscale", "low resolution"),
        ("brightness", "brightness"),
        ("noise", "noise"),
    ]
    fig, axes = plt.subplots(len(effects), 5, figsize=(12, 2.4 * len(effects)))
    for row, (effect, title) in enumerate(effects):
        for sev in range(5):
            y = apply_drone_condition(x.clone(), sev, eval_cfg, effect=effect) if sev else x
            img = y.permute(1, 2, 0).numpy().clip(0, 1)
            axes[row, sev].imshow(img)
            axes[row, sev].set_title(f"{title} s{sev}" if sev == 0 or row == 0 else f"s{sev}")
            axes[row, sev].axis("off")
        axes[row, 0].set_ylabel(title)
    fig.suptitle("Drone capture simulation by effect and severity")
    fig.tight_layout()
    fig.savefig(OUT / "drone_conditions.png", dpi=140)
    plt.close(fig)
    print("wrote", OUT / "drone_conditions.png")
else:
    print("skip drone previews: no sample tile")


## Class distribution per split

In [ ]:
if STATS.is_file():
    stats = json.loads(STATS.read_text())
    pc = stats["per_class"]
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(ANOMALY))
    w = 0.25
    for i, sp in enumerate(["train", "val", "test"]):
        vals = [pc[sp][c]["tile_count"] for c in ANOMALY]
        ax.bar(x + i * w, vals, w, label=sp)
    ax.set_xticks(x + w)
    ax.set_xticklabels(ANOMALY, rotation=45, ha="right")
    ax.set_ylabel("tiles with class")
    ax.legend()
    ax.set_title(f"Class tile counts ({PROFILE})")
    fig.tight_layout()
    fig.savefig(OUT / "class_distribution.png", dpi=140)
    plt.close(fig)
    print("wrote", OUT / "class_distribution.png")
else:
    print("missing", STATS)

## Training curves, confusion, IoU/F1, PR, robustness, efficiency, sample preds

In [ ]:
def load_log(run):
    p = REPO / "runs" / PROFILE / run / "log.csv"
    if not p.is_file():
        return []
    return list(csv.DictReader(p.open()))

def load_metrics(run):
    p = REPO / "runs" / PROFILE / run / "eval" / "metrics.json"
    return json.loads(p.read_text()) if p.is_file() else None

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for run in RUNS:
    rows = [r for r in load_log(run) if float(r.get("epoch_sec", 0)) > 10 or PROFILE != "pilot"]
    if not rows:
        rows = load_log(run)
    if not rows:
        continue
    ep = [int(r["epoch"]) for r in rows]
    axes[0].plot(ep, [float(r["train_loss"]) for r in rows], label=run)
    axes[1].plot(ep, [float(r["val_miou"]) for r in rows], label=run)
axes[0].set_title("train loss"); axes[0].legend()
axes[1].set_title("val mIoU"); axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "training_curves.png", dpi=140)
plt.close(fig)
print("wrote", OUT / "training_curves.png")

# Confusion matrices
for run in RUNS:
    src = REPO / "runs" / PROFILE / run / "eval" / "confusion.png"
    if src.is_file():
        Image.open(src).save(OUT / f"confusion_{run}.png")
        print("copied", OUT / f"confusion_{run}.png")

# Per class IoU and F1 bars
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(ANOMALY))
w = 0.25
for i, run in enumerate(RUNS):
    m = load_metrics(run)
    if not m:
        continue
    iou = [m["per_class_iou_modified"].get(c, 0.0) for c in ANOMALY]
    f1 = [m.get("multilabel", {}).get("f1", {}).get(c, 0.0) for c in ANOMALY]
    axes[0].bar(x + i * w, iou, w, label=run)
    axes[1].bar(x + i * w, f1, w, label=run)
for ax, title in zip(axes, ["per class IoU (modified)", "per class F1 (multilabel)"]):
    ax.set_xticks(x + w); ax.set_xticklabels(ANOMALY, rotation=45, ha="right")
    ax.set_title(title); ax.legend()
fig.tight_layout()
fig.savefig(OUT / "per_class_iou_f1.png", dpi=140)
plt.close(fig)
print("wrote", OUT / "per_class_iou_f1.png")

# PR curves
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, run in zip(axes, RUNS):
    prp = REPO / "runs" / PROFILE / run / "eval" / "pr_curves.json"
    if not prp.is_file():
        ax.set_title(f"{run} missing"); continue
    pr = json.loads(prp.read_text())
    for c in ANOMALY:
        curve = pr.get(c) or {}
        if curve.get("recall"):
            ax.plot(curve["recall"], curve["precision"], label=c, linewidth=1)
    ax.set_title(run); ax.set_xlabel("recall"); ax.set_ylabel("precision")
fig.tight_layout()
fig.savefig(OUT / "pr_curves.png", dpi=140)
plt.close(fig)
print("wrote", OUT / "pr_curves.png")

# Robustness
fig, ax = plt.subplots(figsize=(6, 4))
any_rob = False
for run in RUNS:
    rp = REPO / "runs" / PROFILE / run / "eval" / "robustness.json"
    if not rp.is_file():
        continue
    pts = json.loads(rp.read_text())
    ax.plot([p["severity"] for p in pts], [p["modified_miou"] for p in pts], marker="o", label=run)
    any_rob = True
ax.set_xlabel("severity"); ax.set_ylabel("modified mIoU"); ax.set_title("Robustness")
if any_rob:
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "robustness.png", dpi=140)
    print("wrote", OUT / "robustness.png")
else:
    print("no robustness.json yet")
plt.close(fig)

# Efficiency table image
rows_eff = []
for run in RUNS:
    ep = REPO / "runs" / PROFILE / run / "eval" / "efficiency.json"
    if ep.is_file():
        e = json.loads(ep.read_text())
        rows_eff.append([run, e.get("parameters"), round(e.get("size_mb", 0), 2),
                         round(e.get("mean_infer_sec_gpu", 0), 4),
                         round(e.get("mean_infer_sec_cpu", 0), 4)])
if rows_eff:
    fig, ax = plt.subplots(figsize=(8, 2 + 0.4 * len(rows_eff)))
    ax.axis("off")
    table = ax.table(cellText=rows_eff,
                     colLabels=["run", "params", "size_mb", "infer_s_gpu", "infer_s_cpu"],
                     loc="center")
    table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 1.4)
    fig.tight_layout()
    fig.savefig(OUT / "efficiency_table.png", dpi=140)
    plt.close(fig)
    print("wrote", OUT / "efficiency_table.png")

In [ ]:
# Sample predictions from exported gallery if present
gdir = REPO / "dashboard" / "public" / "data" / "gallery"
index = gdir / "index.json"
if index.is_file():
    tiles = json.loads(index.read_text()).get("tiles") or []
    tiles = tiles[:6]
    if tiles:
        fig, axes = plt.subplots(len(tiles), 5, figsize=(12, 2.2 * len(tiles)))
        if len(tiles) == 1:
            axes = np.array([axes])
        for i, t in enumerate(tiles):
            tid = t.get("id") or t.get("tile_id")
            folder = gdir / tid
            def show(ax, path, title):
                if path.is_file():
                    ax.imshow(Image.open(path))
                ax.set_title(title); ax.axis("off")
            show(axes[i, 0], folder / "rgb.webp", "RGB")
            show(axes[i, 1], folder / "gt.webp", "GT")
            show(axes[i, 2], folder / "pred_run1.webp", "run1")
            show(axes[i, 3], folder / "pred_run2.webp", "run2")
            show(axes[i, 4], folder / "pred_run3.webp", "run3")
        fig.tight_layout()
        fig.savefig(OUT / "sample_predictions.png", dpi=140)
        plt.close(fig)
        print("wrote", OUT / "sample_predictions.png")
    else:
        print("gallery empty")
else:
    print("no gallery index; export with --allow-images first")